# C3 Step 6 — E005: hyperparameter tuning on scripted, frozen test on real (Colab)

**The question this answers:** does adapting the pipeline to our ~47 min of
scripted Sinhala-English data improve a *new, real* recording it never saw —
versus the untouched zero-shot baseline (E004)?

**Design (logbook D10):** the scripted corpus shares the same few speakers, so
a speaker-clean split inside it is impossible. Therefore: **TUNE** on all 25
scripted clips, **TEST frozen** on the 2 real YouTube clips (IRD). n=2 — say
so every time the numbers are shown.

**What is tuned:** clustering `threshold` and `min_cluster_size` ONLY.
The model weights are untouched. Under-counting means the model merges
speakers too eagerly, so the grid is biased toward *lower* thresholds.

**Protocol v1 throughout:** no speaker count supplied, overlap included,
explicit UEM, collars 0.0 and 0.25. The tuned run is a separate, labelled
condition — the zero-shot baseline stays the official "before" forever.
**Never pick parameters by looking at TEST scores.** Cell 8 only runs after
Cell 7 has chosen the winner on TUNE.

## Do these once
1. Same HF setup as Step 2 (accept community-1 terms; Colab secret `HF_TOKEN`).
2. Runtime > Change runtime type > **T4 GPU**.
3. Have `E005_upload.tgz` ready (made for you in `C3 Pilot\`).

Rough cost: ~18 grid passes x ~2-3 min = **45-75 min of T4 time**. Cell 7
checkpoints after every pass — if Colab dies, re-run from Cell 3 and it
resumes where it stopped.


In [ ]:
# Cell 1 — check you got a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Nothing shown? Runtime > Change runtime type > T4 GPU > Save, then re-run.


In [ ]:
# Cell 2 — install (2-3 min)
!pip install -q pyannote.audio pyannote.metrics

print('\n' + '=' * 62)
print('NOW DO THIS:   Runtime  >  Restart session')
print('Then SKIP this cell and carry on from Cell 3.')
print('=' * 62)


In [ ]:
# Cell 3 — upload E005_upload.tgz, then run this cell
# (folder icon on the left -> upload button -> pick E005_upload.tgz)
import os, glob

if not os.path.exists('/content/E005_upload.tgz'):
    raise SystemExit('E005_upload.tgz not found in /content. Upload it first '
                     '(it is in H:\\Research Project\\C3 Pilot\\).')
!cd /content && tar xf E005_upload.tgz

TEST = ['IRD_Harshana_Silva', 'IRD_Recording_2_IIT']   # real audio. FROZEN.
wavs   = sorted(os.path.basename(p)[:-4] for p in glob.glob('/content/processed/*.wav'))
truths = sorted(os.path.basename(p)[:-5] for p in glob.glob('/content/rttm_truth/*.rttm'))
TUNE = [c for c in truths if c not in TEST]

assert wavs == truths, f'wav/truth mismatch: {set(wavs) ^ set(truths)}'
for c in truths:
    assert os.path.exists(f'/content/rttm_truth/{c}.uem'), f'missing UEM: {c}'
for c in TEST:
    assert c in truths, f'missing test clip: {c}'
    assert os.path.exists(f'/content/rttm_draft/{c}.rttm'), f'missing zero-shot draft: {c}'
print(f'{len(TUNE)} tune clips, {len(TEST)} test clips (frozen: {TEST})')


In [ ]:
# Cell 4 — load the model and FIND THE KNOBS
import os
os.environ['PYANNOTE_METRICS_ENABLED'] = '0'   # privacy premise: no telemetry

import torch
from google.colab import userdata
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=userdata.get('HF_TOKEN'))            # the NAME of the secret
if pipeline is None:
    raise SystemExit('Pipeline empty — terms not accepted or token wrong.')
pipeline.to(torch.device('cuda'))

DEFAULTS = pipeline.parameters(instantiated=True)
print('Pipeline parameters as shipped (RECORD THESE IN THE LOGBOOK):')
print(DEFAULTS)

# Locate the clustering knobs without assuming the exact layout.
CL = None
for k, v in DEFAULTS.items():
    if isinstance(v, dict) and 'threshold' in v:
        CL = k
if CL is None:
    raise SystemExit('No clustering threshold found in pipeline.parameters(). '
                     'The 4.x layout changed — paste the printout above into '
                     'the chat and we adapt the grid to it. DO NOT guess.')
THR0 = DEFAULTS[CL]['threshold']
MCS0 = DEFAULTS[CL].get('min_cluster_size')
print(f"\nknobs found under '{CL}': threshold={THR0}, min_cluster_size={MCS0}")


In [ ]:
# Cell 5 — protocol v1 scoring helpers (same math as 05_score.py / E004)
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

def load_rttm(path, uri):
    ann = Annotation(uri=uri)
    for i, line in enumerate(open(path, encoding='utf-8')):
        p = line.split()
        if len(p) >= 8 and p[0] == 'SPEAKER' and float(p[4]) > 0:
            ann[Segment(float(p[3]), float(p[3]) + float(p[4])), f't{i}'] = p[7]
    return ann

def load_uem(path, uri):
    tl = Timeline(uri=uri)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) >= 4: tl.add(Segment(float(p[2]), float(p[3])))
    return tl

TRUTH = {c: (load_rttm(f'/content/rttm_truth/{c}.rttm', c),
             load_uem(f'/content/rttm_truth/{c}.uem', c))
         for c in (TUNE + TEST)}

def score(hyps, clips, collar):
    """hyps: {clip: Annotation}. Returns pooled + per-clip DER% under protocol v1."""
    pooled = dict.fromkeys(('confusion', 'missed detection', 'false alarm', 'total'), 0.0)
    per = {}
    for c in clips:
        ref, uem = TRUTH[c]
        m = DiarizationErrorRate(collar=collar, skip_overlap=False)
        comp = m(ref, hyps[c], uem=uem, detailed=True)
        per[c] = (comp['confusion'] + comp['missed detection']
                  + comp['false alarm']) / comp['total'] * 100
        for k in pooled: pooled[k] += comp[k]
    der = (pooled['confusion'] + pooled['missed detection']
           + pooled['false alarm']) / pooled['total'] * 100
    parts = {k: pooled[k] / pooled['total'] * 100
             for k in ('confusion', 'missed detection', 'false alarm')}
    return der, parts, per


In [ ]:
# Cell 6 — the "before": score the shipped zero-shot drafts on TEST
# Must reproduce E004 (baseline_2026-08-25.csv): 27.99 / 4.87, pooled 7.50 @ collar 0.0.
before_hyps = {c: load_rttm(f'/content/rttm_draft/{c}.rttm', c) for c in TEST}
BEFORE = {}
for collar in (0.0, 0.25):
    der, parts, per = score(before_hyps, TEST, collar)
    BEFORE[collar] = (der, parts, per)
    print(f'BEFORE (zero-shot) collar {collar}: pooled DER {der:.2f}%  per-clip {per}')
print('\nIf these do not match E004 to ~0.01, STOP and check the uploaded files.')


In [ ]:
# Cell 7 — grid search on TUNE only. Checkpointed; safe to re-run after a crash.
import copy, csv, time, os

# Under-counting => merging too much => bias the grid to LOWER thresholds.
THRESHOLDS = [round(THR0 + d, 4) for d in (-0.20, -0.15, -0.10, -0.05, 0.0, +0.05)]
MCS_VALUES = sorted({max(2, MCS0 // 2), max(2, int(MCS0 * 0.75)), MCS0}) if MCS0 else [None]
CKPT = '/content/e005_grid.csv'

done = {}
if os.path.exists(CKPT):
    for r in csv.DictReader(open(CKPT)):
        done[(float(r['threshold']), r['min_cluster_size'])] = float(r['DER_tune_c0.0'])
    print(f'resuming: {len(done)} combos already scored')
else:
    with open(CKPT, 'w', newline='') as fh:
        csv.writer(fh).writerow(['threshold', 'min_cluster_size',
                                 'DER_tune_c0.0', 'conf', 'miss', 'fa', 'DER_tune_c0.25'])

total = len(THRESHOLDS) * len(MCS_VALUES)
n = 0
for mcs in MCS_VALUES:
    for thr in THRESHOLDS:
        n += 1
        key = (thr, str(mcs))
        if key in done:
            continue
        params = copy.deepcopy(DEFAULTS)
        params[CL]['threshold'] = thr
        if mcs is not None: params[CL]['min_cluster_size'] = mcs
        pipeline.instantiate(params)
        t0 = time.time()
        hyps = {c: pipeline(f'/content/processed/{c}.wav').speaker_diarization
                for c in TUNE}
        d0, parts, _ = score(hyps, TUNE, 0.0)
        d25, _, _ = score(hyps, TUNE, 0.25)
        with open(CKPT, 'a', newline='') as fh:
            csv.writer(fh).writerow([thr, mcs, f'{d0:.4f}', f"{parts['confusion']:.4f}",
                                     f"{parts['missed detection']:.4f}",
                                     f"{parts['false alarm']:.4f}", f'{d25:.4f}'])
        done[key] = d0
        print(f'[{n}/{total}] thr={thr} mcs={mcs}  DER(tune) {d0:.2f}%  '
              f'conf {parts["confusion"]:.2f}  ({time.time()-t0:.0f}s)')

rows = sorted(csv.DictReader(open(CKPT)), key=lambda r: float(r['DER_tune_c0.0']))
print('\n=== grid, best first (chosen on TUNE, collar 0.0) ===')
for r in rows[:8]:
    print(f"thr={r['threshold']} mcs={r['min_cluster_size']}  "
          f"DER {float(r['DER_tune_c0.0']):.2f}  conf {float(r['conf']):.2f}")
BEST = rows[0]
print(f"\nWINNER: threshold={BEST['threshold']} min_cluster_size={BEST['min_cluster_size']}")


In [ ]:
# Cell 8 — freeze the winner, run ONCE on the 2 real TEST clips
import copy, os
params = copy.deepcopy(DEFAULTS)
params[CL]['threshold'] = float(BEST['threshold'])
if BEST['min_cluster_size'] not in ('None', ''):
    params[CL]['min_cluster_size'] = int(BEST['min_cluster_size'])
pipeline.instantiate(params)

os.makedirs('/content/rttm_tuned', exist_ok=True)
after_hyps = {}
for c in TEST:
    ann = pipeline(f'/content/processed/{c}.wav').speaker_diarization
    after_hyps[c] = ann
    with open(f'/content/rttm_tuned/{c}.rttm', 'w') as fh:
        ann.write_rttm(fh)

print('E005 RESULT — zero-shot vs tuned, on real audio the tuning never saw (n=2!)')
for collar in (0.0, 0.25):
    b_der, b_parts, b_per = BEFORE[collar]
    a_der, a_parts, a_per = score(after_hyps, TEST, collar)
    print(f'\ncollar {collar}:')
    print(f"  {'clip':<22}{'before':>9}{'after':>9}{'change':>9}")
    for c in TEST:
        print(f'  {c:<22}{b_per[c]:>9.2f}{a_per[c]:>9.2f}{a_per[c]-b_per[c]:>+9.2f}')
    print(f"  {'POOLED':<22}{b_der:>9.2f}{a_der:>9.2f}{a_der-b_der:>+9.2f}"
          f"   (conf {b_parts['confusion']:.2f} -> {a_parts['confusion']:.2f})")

# also record what the winner did on its own tuning set (in-domain, leaky by design)
d0, parts, _ = score({c: pipeline(f'/content/processed/{c}.wav').speaker_diarization
                      for c in TUNE}, TUNE, 0.0)
print(f'\nTuned on TUNE set itself (in-domain, optimistic): DER {d0:.2f}% @ 0.0'
      f'  (zero-shot E004 scripted pooled was ~19.9%)')


In [ ]:
# Cell 9 — download everything for the logbook
import json
summary = {
    'experiment': 'E005', 'design': 'tune scripted (n=25) / test real IRD (n=2)',
    'model': 'pyannote/speaker-diarization-community-1',
    'tuned_params': {'group': CL, 'threshold': float(BEST['threshold']),
                     'min_cluster_size': BEST['min_cluster_size']},
    'defaults': str(DEFAULTS),
    'protocol': 'v1: no speaker count, overlap included, explicit UEM, collars 0.0/0.25',
}
import pyannote.audio, pyannote.metrics
summary['pyannote.audio'] = pyannote.audio.__version__
summary['pyannote.metrics'] = pyannote.metrics.__version__
with open('/content/e005_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=1)

!cd /content && zip -qr e005_results.zip e005_grid.csv e005_summary.json rttm_tuned
from google.colab import files
files.download('e005_results.zip')

# Unzip into  H:\Research Project\C3 Pilot\results\E005\
# Then: append the numbers + the winning parameters + library versions to
# C3_LOGBOOK.md (there is a placeholder entry dated 25 Aug waiting for them).
